# PINN Project 4: Inverse Problem — Discovering Unknown Parameters
## The Most Powerful Application of PINNs

### What is an Inverse Problem?
In **forward problems** (Projects 1-3), we know the PDE and solve for the solution $u$.

In **inverse problems**, we have some **measurement data** and want to discover
**unknown physical parameters** in the PDE — for example:
- What is the thermal conductivity of this material?
- What is the diffusion coefficient in this fluid?
- What is the damping rate of this oscillator?

### Problem Description
We solve the **1D diffusion equation** with **unknown diffusion coefficient** $D$:

$$\frac{\partial u}{\partial t} = D \frac{\partial^2 u}{\partial x^2}$$

- **True value**: $D = 0.1$ (unknown to the PINN — it must discover this!)
- Domain: $x \in [0, 1]$, $t \in [0, 1]$
- Initial condition: $u(x, 0) = \sin(\pi x)$
- Boundary conditions: $u(0, t) = u(1, t) = 0$
- **Exact solution**: $u(x, t) = \sin(\pi x) \cdot \exp(-D \pi^2 t)$

### How does the PINN discover D?
We make $D$ a **trainable parameter** in the network. The PINN simultaneously:
1. Learns the solution $u(x, t)$ (like before)
2. Learns the parameter $D$ (new!)

The measurement data constrains the network, and the PDE structure
forces $D$ to converge to its true value.


In [ ]:
# ============================================================
# STEP 1: Import Libraries
# ============================================================
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')


In [ ]:
# ============================================================
# STEP 2: Generate Synthetic "Measurement" Data
# ============================================================
# In a real problem, this data would come from sensors/experiments.
# Here, we generate it from the exact solution to verify our method.

D_true = 0.1  # True diffusion coefficient (the PINN must discover this!)

def exact_solution(x, t, D):
    """Exact solution: u(x,t) = sin(πx) * exp(-D*π²*t)"""
    return np.sin(np.pi * x) * np.exp(-D * np.pi**2 * t)

# Generate measurement data at random locations
N_data = 200  # Number of measurement points

# Random measurement locations inside the domain
x_data_np = np.random.uniform(0, 1, N_data)
t_data_np = np.random.uniform(0, 1, N_data)
u_data_np = exact_solution(x_data_np, t_data_np, D_true)

# Add 2% Gaussian noise to simulate real measurements
noise_level = 0.02
u_data_np += noise_level * np.std(u_data_np) * np.random.randn(N_data)

# Convert to tensors
x_data = torch.tensor(x_data_np, dtype=torch.float32).reshape(-1, 1).to(device)
t_data = torch.tensor(t_data_np, dtype=torch.float32).reshape(-1, 1).to(device)
u_data = torch.tensor(u_data_np, dtype=torch.float32).reshape(-1, 1).to(device)

print(f"Generated {N_data} noisy measurement points")
print(f"True D = {D_true}")
print(f"Noise level: {noise_level*100:.0f}%")
print(f"\nThe PINN will try to discover D from this data + physics!")

# Visualize the measurement data
fig, ax = plt.subplots(figsize=(8, 6))
sc = ax.scatter(x_data_np, t_data_np, c=u_data_np, cmap='RdBu_r', s=20, alpha=0.7)
plt.colorbar(sc, label='u (measured)')
ax.set_xlabel('x')
ax.set_ylabel('t')
ax.set_title(f'Measurement Data ({N_data} points with {noise_level*100:.0f}% noise)')
plt.show()


In [ ]:
# ============================================================
# STEP 3: Define the Network with TRAINABLE Parameter D
# ============================================================

class PINN_Inverse(nn.Module):
    """
    PINN for inverse problem: discovers the diffusion coefficient D.
    
    KEY DIFFERENCE from forward problems:
    - D is a nn.Parameter — it gets updated during training just like weights!
    - We initialize D with a wrong guess (0.5) and let the network learn it
    
    Input:  (x, t) — 2 values
    Output: u(x,t) — 1 value
    Trainable parameter: D (diffusion coefficient)
    """
    
    def __init__(self, n_hidden=4, n_neurons=64, D_init=0.5):
        """
        Args:
            D_init: Initial guess for D (intentionally wrong!)
        """
        super().__init__()
        
        # The unknown parameter D — registered as a trainable parameter
        # nn.Parameter tells PyTorch to include this in optimization
        # We use log(D) to ensure D > 0 (physical constraint)
        self.log_D = nn.Parameter(torch.tensor(np.log(D_init), dtype=torch.float32))
        
        # Build the neural network (same as Projects 1-3)
        layers = []
        layers.append(nn.Linear(2, n_neurons))
        layers.append(nn.Tanh())
        for _ in range(n_hidden - 1):
            layers.append(nn.Linear(n_neurons, n_neurons))
            layers.append(nn.Tanh())
        layers.append(nn.Linear(n_neurons, 1))
        self.network = nn.Sequential(*layers)
        
        # Xavier initialization
        for m in self.network:
            if isinstance(m, nn.Linear):
                nn.init.xavier_normal_(m.weight)
                nn.init.zeros_(m.bias)
    
    @property
    def D(self):
        """Get the current estimate of D (always positive due to exp)."""
        return torch.exp(self.log_D)
    
    def forward(self, x, t):
        inputs = torch.cat([x, t], dim=1)
        return self.network(inputs)

# Create model with initial guess D=0.5 (true value is 0.1)
model = PINN_Inverse(n_hidden=4, n_neurons=64, D_init=0.5).to(device)

print(f"Initial D estimate: {model.D.item():.4f}")
print(f"True D value:       {D_true}")
print(f"\nThe PINN will learn to correct D from 0.5 to ~0.1")
print(f"\nTotal network parameters: {sum(p.numel() for p in model.network.parameters()):,}")
print(f"Trainable physics parameter: D (log_D)")


In [ ]:
# ============================================================
# STEP 4: Physics + Data Loss
# ============================================================

def compute_pde_residual(model, x, t):
    """
    Diffusion equation residual: du/dt - D * d²u/dx² = 0
    
    Note: D is now model.D — a trainable parameter!
    """
    u = model(x, t)
    
    # du/dt
    du_dt = torch.autograd.grad(
        u, t, grad_outputs=torch.ones_like(u), create_graph=True
    )[0]
    
    # du/dx
    du_dx = torch.autograd.grad(
        u, x, grad_outputs=torch.ones_like(u), create_graph=True
    )[0]
    
    # d²u/dx²
    d2u_dx2 = torch.autograd.grad(
        du_dx, x, grad_outputs=torch.ones_like(du_dx), create_graph=True
    )[0]
    
    # PDE residual: du/dt - D * d²u/dx² = 0
    # model.D is the TRAINABLE parameter that the network learns
    residual = du_dt - model.D * d2u_dx2
    
    return residual


def compute_loss(model, x_pde, t_pde, x_data, t_data, u_data,
                 x_ic, t_ic, u_ic, x_bc, t_bc, u_bc):
    """
    Total loss for inverse problem:
    
    Loss = PDE_loss + DATA_loss + IC_loss + BC_loss
    
    DATA_loss is the KEY new component:
    - It measures how well the PINN matches the measurement data
    - This is what allows the network to identify the unknown parameter D
    - Without data, D is unconstrained and the inverse problem is ill-posed
    """
    # 1. PDE loss (physics constraint)
    residual = compute_pde_residual(model, x_pde, t_pde)
    loss_pde = torch.mean(residual ** 2)
    
    # 2. DATA loss (measurement data — this drives parameter identification!)
    u_pred_data = model(x_data, t_data)
    loss_data = torch.mean((u_pred_data - u_data) ** 2)
    
    # 3. Initial condition loss
    u_pred_ic = model(x_ic, t_ic)
    loss_ic = torch.mean((u_pred_ic - u_ic) ** 2)
    
    # 4. Boundary condition loss
    u_pred_bc = model(x_bc, t_bc)
    loss_bc = torch.mean((u_pred_bc - u_bc) ** 2)
    
    # Weighted sum
    w_pde = 1.0
    w_data = 10.0    # Data is crucial for inverse problems
    w_ic = 10.0
    w_bc = 10.0
    
    total = w_pde * loss_pde + w_data * loss_data + w_ic * loss_ic + w_bc * loss_bc
    
    return total, loss_pde, loss_data, loss_ic, loss_bc

print("Loss functions defined!")
print("\nKey difference: DATA loss constrains the unknown parameter D")


In [ ]:
# ============================================================
# STEP 5: Prepare Training Points
# ============================================================

N_pde = 5000   # Interior collocation points
N_ic = 200     # Initial condition points
N_bc = 200     # Boundary points

# --- PDE collocation points ---
x_pde = torch.rand(N_pde, 1, device=device, requires_grad=True)
t_pde = torch.rand(N_pde, 1, device=device, requires_grad=True)

# --- Initial condition: u(x, 0) = sin(πx) ---
x_ic = torch.rand(N_ic, 1, device=device)
t_ic = torch.zeros(N_ic, 1, device=device)
u_ic = torch.sin(np.pi * x_ic)

# --- Boundary conditions: u(0, t) = u(1, t) = 0 ---
t_bc_vals = torch.rand(N_bc, 1, device=device)

x_bc_left = torch.zeros(N_bc // 2, 1, device=device)
x_bc_right = torch.ones(N_bc // 2, 1, device=device)
x_bc = torch.cat([x_bc_left, x_bc_right])
t_bc = torch.cat([t_bc_vals[:N_bc//2], t_bc_vals[N_bc//2:]])
u_bc = torch.zeros(N_bc, 1, device=device)

print(f"PDE points:         {N_pde}")
print(f"Measurement points: {N_data}")
print(f"IC points:          {N_ic}")
print(f"BC points:          {N_bc}")


In [ ]:
# ============================================================
# STEP 6: Training Loop — Watch D converge!
# ============================================================

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3000, gamma=0.5)

n_epochs = 15000
print_every = 1500

history = {'total': [], 'pde': [], 'data': [], 'ic': [], 'bc': [], 'D': []}

print(f"Starting training... (True D = {D_true})")
print("=" * 80)

for epoch in range(1, n_epochs + 1):
    model.train()
    optimizer.zero_grad()
    
    # Resample PDE points
    x_pde_train = torch.rand(N_pde, 1, device=device, requires_grad=True)
    t_pde_train = torch.rand(N_pde, 1, device=device, requires_grad=True)
    
    total_loss, pde_loss, data_loss, ic_loss, bc_loss = compute_loss(
        model, x_pde_train, t_pde_train,
        x_data, t_data, u_data,
        x_ic, t_ic, u_ic,
        x_bc, t_bc, u_bc
    )
    
    total_loss.backward()
    optimizer.step()
    scheduler.step()
    
    # Record history
    history['total'].append(total_loss.item())
    history['pde'].append(pde_loss.item())
    history['data'].append(data_loss.item())
    history['ic'].append(ic_loss.item())
    history['bc'].append(bc_loss.item())
    history['D'].append(model.D.item())
    
    if epoch % print_every == 0 or epoch == 1:
        D_est = model.D.item()
        D_err = abs(D_est - D_true) / D_true * 100
        print(f"Epoch {epoch:5d}/{n_epochs} | "
              f"Loss: {total_loss.item():.2e} | "
              f"D_pred: {D_est:.5f} | "
              f"D_true: {D_true} | "
              f"Error: {D_err:.2f}%")

print("=" * 80)
D_final = model.D.item()
print(f"\nFinal D estimate: {D_final:.6f}")
print(f"True D value:     {D_true}")
print(f"Relative error:   {abs(D_final - D_true)/D_true*100:.2f}%")


In [ ]:
# ============================================================
# STEP 7: Visualize Results
# ============================================================

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# --- Plot 1: Parameter D Convergence ---
axes[0, 0].plot(history['D'], 'b-', linewidth=2, label='D estimate')
axes[0, 0].axhline(y=D_true, color='r', linestyle='--', linewidth=2, label=f'True D = {D_true}')
axes[0, 0].axhline(y=0.5, color='gray', linestyle=':', alpha=0.5, label='Initial guess (0.5)')
axes[0, 0].set_xlabel('Epoch', fontsize=12)
axes[0, 0].set_ylabel('D', fontsize=12)
axes[0, 0].set_title('Parameter D Convergence', fontsize=14)
axes[0, 0].legend(fontsize=11)
axes[0, 0].grid(True, alpha=0.3)

# --- Plot 2: Loss History ---
axes[0, 1].semilogy(history['total'], label='Total', alpha=0.7)
axes[0, 1].semilogy(history['pde'], label='PDE', alpha=0.7)
axes[0, 1].semilogy(history['data'], label='Data', alpha=0.7)
axes[0, 1].semilogy(history['ic'], label='IC', alpha=0.7)
axes[0, 1].semilogy(history['bc'], label='BC', alpha=0.7)
axes[0, 1].set_xlabel('Epoch', fontsize=12)
axes[0, 1].set_ylabel('Loss', fontsize=12)
axes[0, 1].set_title('Training Loss History', fontsize=14)
axes[0, 1].legend(fontsize=10)
axes[0, 1].grid(True, alpha=0.3)

# --- Plot 3: PINN Prediction vs Exact ---
nx, nt = 100, 100
x_grid = np.linspace(0, 1, nx)
t_grid = np.linspace(0, 1, nt)
X, T = np.meshgrid(x_grid, t_grid)

x_flat = torch.tensor(X.flatten(), dtype=torch.float32).reshape(-1, 1).to(device)
t_flat = torch.tensor(T.flatten(), dtype=torch.float32).reshape(-1, 1).to(device)

model.eval()
with torch.no_grad():
    u_pred = model(x_flat, t_flat).cpu().numpy().reshape(nt, nx)

u_exact = exact_solution(X, T, D_true)

c3 = axes[1, 0].contourf(X, T, u_pred, levels=50, cmap='RdBu_r')
plt.colorbar(c3, ax=axes[1, 0])
axes[1, 0].scatter(x_data_np, t_data_np, c='black', s=5, alpha=0.3, label='Data points')
axes[1, 0].set_xlabel('x', fontsize=12)
axes[1, 0].set_ylabel('t', fontsize=12)
axes[1, 0].set_title('PINN Prediction u(x,t)', fontsize=14)
axes[1, 0].legend(fontsize=10)

# --- Plot 4: Error ---
error = np.abs(u_pred - u_exact)
c4 = axes[1, 1].contourf(X, T, error, levels=50, cmap='hot_r')
plt.colorbar(c4, ax=axes[1, 1])
axes[1, 1].set_xlabel('x', fontsize=12)
axes[1, 1].set_ylabel('t', fontsize=12)
axes[1, 1].set_title(f'Absolute Error (max = {error.max():.2e})', fontsize=14)

plt.tight_layout()
plt.show()

# --- Time slice comparison ---
fig2, axes2 = plt.subplots(1, 3, figsize=(18, 5))
for i, t_val in enumerate([0.0, 0.5, 1.0]):
    t_idx = int(t_val * (nt - 1))
    axes2[i].plot(x_grid, u_exact[t_idx, :], 'b-', linewidth=2, label='Exact')
    axes2[i].plot(x_grid, u_pred[t_idx, :], 'r--', linewidth=2, label='PINN')
    axes2[i].set_xlabel('x', fontsize=12)
    axes2[i].set_ylabel('u(x,t)', fontsize=12)
    axes2[i].set_title(f't = {t_val:.1f}', fontsize=14)
    axes2[i].legend(fontsize=11)
    axes2[i].grid(True, alpha=0.3)

plt.suptitle(f'Solution Comparison (D_pred = {model.D.item():.5f}, D_true = {D_true})', 
             fontsize=14, y=1.02)
plt.tight_layout()
plt.show()


## Key Takeaways

### What makes Inverse PINNs special?

1. **Parameter as a trainable variable**: `nn.Parameter(torch.tensor(log_D))` makes D 
   optimizable alongside the network weights.

2. **Log-parameterization**: We train `log(D)` instead of `D` directly to ensure 
   $D > 0$ (physical constraint: diffusion coefficient must be positive).

3. **Data is essential**: Without measurement data, the parameter D is unconstrained. 
   The data loss is what drives D to its true value.

4. **Noisy data works**: Even with 2% noise, the PINN recovers D accurately because 
   the PDE constraint acts as a **regularizer** — it filters out noise.

5. **Simultaneous learning**: The PINN learns BOTH the solution u(x,t) AND the parameter D 
   in a single training process.




